<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# A Midbody of Three Ends: Is the One-Point Jitter the Cause?

The check written down in the journal of 2026-10-15, *A Midbody of Three Ends*, before it ran: on the seventeen records
of the speed check, the signed excess of frames in no state, the lab's minus the published, per record, for the
published-choice speed with the midbody as the middle end of the eight arcs and as the mean of the three middle ends.
The jitter reading predicts the three-end midbody lowers the median excess (+0.058 with one end) by at least half, to
+0.029 or less; a fall of less than a quarter, to more than +0.0435, refutes it; anything between is inconclusive. The
states check's bar is reported for the three-end midbody as a description, not as a test of the reading.

Run from the repository's root with the block-1 files in `data/raw/owmd-n2/` and the Rust toolchain installed.


In [1]:
import os, subprocess
import numpy as np
import h5py

os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DIRECTORY, SUFFIX = "data/raw/owmd-n2", "_features.hdf5"
# The seventeen records of the speed check of 2026-10-14, pinned so that a re-run checks the same set.
records = ["1006562", "1007123", "1007405", "1008698", "1011246", "1011901", "1013148", "1014831", "1020507",
           "1021835", "1024910", "1025254", "1025397", "1029332", "1030984", "1031252", "1032937"]
missing = [record for record in records if not os.path.exists(f"{DIRECTORY}/{record}{SUFFIX}")]
assert not missing, f"fetch these first: {missing}"
subprocess.run(["cargo", "build", "--quiet", "--release", "--bin", "locomotion-states"], check=True)
print(len(records), "records checked")
print("h5py", h5py.__version__, "numpy", np.__version__)

17 records checked
h5py 3.16.0 numpy 2.5.4


In [2]:
STATES = {"forward": 1, "backward": -1, "paused": 0}

def lab_states(record, flag):
    out = subprocess.run(["target/release/locomotion-states", flag, record], check=True, capture_output=True,
                         text=True).stdout
    return np.array([float(line) for line in out.splitlines()])

published = {}
for record in records:
    with h5py.File(f"{DIRECTORY}/{record}{SUFFIX}", "r") as file:
        published[record] = file["/features_timeseries"]["motion_modes"].astype(float)

def share(modes, value):
    return np.mean(np.isnan(modes)) if value is None else np.mean(modes == value)

theirs = {name: np.array([share(published[r], v) for r in records]) for name, v in STATES.items()}
spread = {name: np.std(theirs[name], ddof=1) for name in STATES}

for flag, title in (("--published-choices", "one end (the middle end)"), ("--three-ends", "three middle ends")):
    ours = {r: lab_states(r, flag) for r in records}
    for r in records:
        assert len(ours[r]) == len(published[r]), (r, len(ours[r]), len(published[r]))
    excess = np.array([share(ours[r], None) - share(published[r], None) for r in records])
    print(title)
    print(f"  excess of frames in no state: median {np.median(excess):+.4f}, range {excess.min():+.4f} to "
          f"{excess.max():+.4f}, higher in {np.sum(excess > 0)} of {len(records)}")
    for name, v in STATES.items():
        diff = np.array([share(ours[r], v) for r in records]) - theirs[name]
        median, worst = np.median(np.abs(diff)), np.max(np.abs(diff))
        ok = median <= spread[name] / 4 and worst <= spread[name] / 2
        print(f"  {name:8s} median |diff| {median:.4f} (bar {spread[name] / 4:.4f})  max |diff| {worst:.4f} "
              f"(bar {spread[name] / 2:.4f})  within the bar {ok}  signed median {np.median(diff):+.4f}")
    print()

one end (the middle end)
  excess of frames in no state: median +0.0580, range +0.0172 to +0.1150, higher in 17 of 17
  forward  median |diff| 0.0093 (bar 0.0598)  max |diff| 0.0228 (bar 0.1196)  within the bar True  signed median -0.0093
  backward median |diff| 0.0028 (bar 0.0093)  max |diff| 0.0111 (bar 0.0186)  within the bar True  signed median -0.0028
  paused   median |diff| 0.0473 (bar 0.0511)  max |diff| 0.0842 (bar 0.1022)  within the bar True  signed median -0.0473



three middle ends
  excess of frames in no state: median +0.0669, range +0.0216 to +0.1227, higher in 17 of 17
  forward  median |diff| 0.0297 (bar 0.0598)  max |diff| 0.0511 (bar 0.1196)  within the bar True  signed median -0.0297
  backward median |diff| 0.0099 (bar 0.0093)  max |diff| 0.0219 (bar 0.0186)  within the bar False  signed median -0.0099
  paused   median |diff| 0.0297 (bar 0.0511)  max |diff| 0.0647 (bar 0.1022)  within the bar True  signed median -0.0297

